# Searching pointcloud.org with `pystac-client`

[pointcloud.org](https://pointcloud.org) is a public archive of lidar point clouds,
every dataset described as [STAC](https://stacspec.org/) and stored as
[COPC](https://copc.io) — a LAZ variant you can read a piece of over HTTP without
downloading the whole file.

This notebook uses [`pystac-client`](https://pystac-client.readthedocs.io/), the
reference Python STAC client, against the archive's `/search` endpoint. Everything
here is a live query — no credentials, no signup, no download until you ask for one.

**What you'll do:** open the API, list what's in it, filter by space, time and point
count, then pull a real COPC URL out of the results and read the file's header
without fetching the file.

## Setup

`pystac-client` is the only requirement for the search parts. The last two sections
optionally use `duckdb` (for bulk queries against the archive's GeoParquet) and
`pdal` (to read a point cloud) — both are called out where they're needed.

In [1]:
# pip install pystac-client
import pystac_client, pystac
print("pystac-client", pystac_client.__version__)
print("pystac       ", pystac.__version__)

pystac-client 0.9.0
pystac        1.15.2


## 1. Open the API

`Client.open()` fetches the landing page and reads its `conformsTo` list, which is
how the client decides what it's allowed to do — whether it can search at all,
whether it can use CQL2 filters, and so on.

In [2]:
from pystac_client import Client

API = "https://pointcloud.org/stac"
client = Client.open(API)

print(f"{client.id}  —  {client.title}")
print()
for uri in sorted(client.get_conforms_to()):
    print(" ", uri)

pointcloud-org  —  pointcloud.org STAC API

  http://www.opengis.net/spec/cql2/1.0/conf/advanced-comparison-operators
  http://www.opengis.net/spec/cql2/1.0/conf/basic-cql2
  http://www.opengis.net/spec/cql2/1.0/conf/cql2-json
  http://www.opengis.net/spec/cql2/1.0/conf/cql2-text
  http://www.opengis.net/spec/ogcapi-features-1/1.0/conf/core
  http://www.opengis.net/spec/ogcapi-features-1/1.0/conf/geojson
  http://www.opengis.net/spec/ogcapi-features-1/1.0/conf/oas30
  http://www.opengis.net/spec/ogcapi-features-3/1.0/conf/features-filter
  http://www.opengis.net/spec/ogcapi-features-3/1.0/conf/filter
  https://api.stacspec.org/v1.0.0/collections
  https://api.stacspec.org/v1.0.0/core
  https://api.stacspec.org/v1.0.0/item-search
  https://api.stacspec.org/v1.0.0/item-search#filter
  https://api.stacspec.org/v1.0.0/ogcapi-features


## 2. What's in the archive?

Each dataset is a STAC **Collection**; each file within it is an **Item**. Most
datasets here hold a single Item, since one lidar project is usually published as
one COPC file.

In [3]:
collections = list(client.get_collections())
print(f"{len(collections)} collections\n")

for c in collections[:8]:
    interval = c.extent.temporal.intervals[0]
    when = interval[0].date().isoformat() if interval[0] else "unknown"
    print(f"  {c.id:28} {when}  {c.title}")
print(f"  ... and {len(collections) - 8} more")

53 collections

  autzen                       2010-06-23  Autzen Stadium
  autzen-2023                  2023-07-21  Autzen Stadium — Willamette Valley 2023 Lidar
  barringer-meteorite-crater   2010-03-12  Barringer Meteorite Crater
  chicago-downtown             2017-04-16  Chicago Downtown
  copenhagen-kastellet         2023-07-12  Copenhagen Kastellet
  IA_Eastern_1_2019            2019-12-10  IA Eastern 1 2019 Lidar
  IA_Eastern_2_2019            2019-12-07  IA Eastern 2 2019 Lidar
  IA_Eastern_3_2019            2019-12-08  IA Eastern 3 2019 Lidar
  ... and 45 more


## 3. Your first search

`search()` builds the query; nothing is sent until you iterate. `max_items` caps how
much the client will page through, which is what you want while exploring.

In [4]:
search = client.search(max_items=5)

for item in search.items():
    props = item.properties
    print(f"{item.id[:40]:42} {props.get('pc:count', 0):>16,} points")

autzen-classified                                    61,201 points
autzen-2023                                         531,361 points
barringer-meteorite-crater-ncalm-2010-10             46,205 points
chicago-downtown                                    358,197 points
copenhagen-kastellet                                177,886 points


## 4. Search by space

`bbox` takes `[west, south, east, north]` in WGS84 degrees.

One caveat worth knowing: this API implements spatial filtering as a **bounding-box
intersection**, not true geometry intersection. A result's footprint is guaranteed to
overlap your box, but for an irregular footprint the box may overlap where the actual
points do not. For precise work, intersect `item.geometry` yourself.

In [5]:
# Roughly the state of Oregon
oregon = [-124.6, 41.9, -116.4, 46.3]

for item in client.search(bbox=oregon).items():
    print(f"{item.id:26} {item.collection_id:20} {item.bbox}")

autzen-classified          autzen               [-123.0755365, 44.04973358, -123.0619606, 44.06277128]
autzen-2023                autzen-2023          [-123.0741187, 44.05008109, -123.0619959, 44.06679313]


## 5. Search by time

`datetime` accepts a single instant, a closed interval, or an open one (`"2020-01-01/.."`).

Note that many datasets in this archive describe a survey that spans weeks, so they
carry `start_datetime`/`end_datetime` and a null `datetime`. The API matches on the
instant where there is one; the two-bound form is what a range search needs.

In [6]:
recent = client.search(datetime="2019-01-01/2021-12-31")
items = list(recent.items())
print(f"{len(items)} items acquired 2019–2021\n")

for item in items[:6]:
    p = item.properties
    when = p.get("datetime") or f"{p.get('start_datetime')} … {p.get('end_datetime')}"
    print(f"  {item.id[:30]:32} {when}")

21 items acquired 2019–2021

  IA_Eastern_1_2019                2019-12-10T00:00:00Z
  IA_Eastern_2_2019                2019-12-07T00:00:00Z
  IA_Eastern_3_2019                2019-12-08T00:00:00Z
  IA_NorthCentral_1_2020           2020-04-04T00:00:00Z
  IA_NorthCentral_2_2020           2020-04-16T00:00:00Z
  IA_NorthCentral_3_2020           2020-04-04T00:00:00Z


## 6. Search by anything else, with CQL2

The interesting queries for point clouds are about the *points*: how many, what kind,
what coordinate system. Those live in Item properties from the
[Point Cloud](https://github.com/stac-extensions/pointcloud) and
[Projection](https://github.com/stac-extensions/projection) STAC extensions, and CQL2
can filter on them.

`/queryables` tells you which properties are filterable and what type each one is.

In [7]:
import json, urllib.request

req = urllib.request.Request(f"{API}/queryables", headers={"User-Agent": "notebook"})
queryables = json.load(urllib.request.urlopen(req))
for name, schema in list(queryables["properties"].items()):
    print(f"  {name:22} {schema.get('type', schema.get('format', ''))}")

  id                     string
  collection             string
  datetime               string
  start_datetime         string
  end_datetime           string
  title                  string
  pc:count               integer
  pc:schemas             string
  pc:statistics          string
  pc:type                string
  proj:epsg              string
  proj:bbox              string
  proj:geometry          string
  proj:projjson          string
  proj:wkt2              string
  stac_extensions        string
  stac_version           string
  geometry               string
  geometry_bbox          string
  bbox                   string
  links                  string
  assets                 string


In [8]:
# Datasets with more than five billion points
big = client.search(
    filter_lang="cql2-json",
    filter={"op": ">", "args": [{"property": "pc:count"}, 5_000_000_000]},
)
items = sorted(big.items(), key=lambda i: -i.properties["pc:count"])
print(f"{len(items)} datasets over 5 billion points\n")
for item in items[:8]:
    print(f"  {item.id[:34]:36} {item.properties['pc:count']:>16,}")

39 datasets over 5 billion points

  MN_RainyLake_1_2020                   355,339,247,767
  MN_UpperMissRiver_6_B22               325,278,739,823
  MN_RainyLake_2_2020                   293,472,666,706
  MN_RiverWest_1_B23                    288,918,266,196
  MN_RiverWest_2_B23                    264,904,992,691
  MN_CentralMissRiver_4_B22             215,065,503,920
  MN_UpperMissRiver_5_B22               200,558,106,069
  MN_GoodhueCo_1_2020                   183,448,445,150


CQL2 also has a text form, which is easier to type by hand and equally valid:

In [9]:
query = "pc:count BETWEEN 40000000000 AND 100000000000"
mid = list(client.search(filter_lang="cql2-text", filter=query).items())
print(f"{len(mid)} datasets between 40 and 100 billion points")
for item in mid[:5]:
    print(f"  {item.id[:34]:36} {item.properties['pc:count']:>16,}")

23 datasets between 40 and 100 billion points
  IA_Eastern_2_2019                      90,121,793,878
  IA_Eastern_3_2019                      99,896,983,309
  IA_NorthCentral_1_2020                 49,774,341,127
  IA_NorthCentral_2_2020                 60,040,043,143
  IA_NorthCentral_3_2020                 71,831,674,676


Filters combine, so you can ask a real question — *large surveys in Minnesota,
acquired since 2020* — in one query:

In [10]:
combined = client.search(
    # Minnesota, give or take: this box also clips the northern edge of Iowa, and
    # you can see that in the results -- the bbox approximation from section 4 in
    # practice.
    bbox=[-97.3, 43.4, -89.4, 49.4],
    datetime="2020-01-01/..",
    filter_lang="cql2-json",
    filter={"op": ">", "args": [{"property": "pc:count"}, 50_000_000_000]},
)
for item in combined.items():
    print(f"  {item.id:28} {item.properties['pc:count']:>16,}")

  IA_Western_2_2020              78,997,864,806
  MN_BeckerCo_1_2021             97,225,507,932
  MN_CentralMissRiver_1_B22      58,846,044,528
  MN_CentralMissRiver_2_B22      94,449,189,320
  MN_CentralMissRiver_3_B22     102,390,618,633
  MN_CentralMissRiver_4_B22     215,065,503,920
  MN_CentralMissRiver_5_B22      83,069,293,873
  MN_CentralMissRiver_6_B22      85,796,446,582
  MN_GoodhueCo_1_2020           183,448,445,150
  MN_LakeSuperior_1_2021         98,942,509,604


  MN_LakeSuperior_2_2021        179,028,497,838
  MN_MORiverBigSioux_1_B21       91,332,165,862
  MN_RainyLake_1_2020           355,339,247,767
  MN_RainyLake_2_2020           293,472,666,706
  MN_RiverEast_1_B23             70,023,749,514
  MN_RiverWest_1_B23            288,918,266,196
  MN_RiverWest_2_B23            264,904,992,691
  MN_RiverWest_3_B23            104,729,861,220
  MN_SEDriftless_1_2021          61,930,563,511
  MN_SEDriftless_2_2021         102,364,344,101


  MN_SEDriftless_3_2021          67,476,190,327
  MN_SEDriftless_4_2021          73,144,772,767
  MN_SEDriftless_5_2021          80,744,226,841
  MN_UpperMissRiver_2_B22       180,708,720,683
  MN_UpperMissRiver_4_B22       180,298,113,067
  MN_UpperMissRiver_5_B22       200,558,106,069
  MN_UpperMissRiver_6_B22       325,278,739,823


## 7. Inside an Item

An Item carries the metadata you need to decide whether to download anything: how many
points, what classification schema, which coordinate reference system, and the asset
URL itself.

In [11]:
item = next(client.search(collections=["autzen"]).items())

print(item.id, "in", item.collection_id)
print()
print("assets:")
for key, asset in item.assets.items():
    print(f"  {key:12} {asset.media_type}")
    print(f"               {asset.href}")
print()
p = item.properties
print("points        :", f"{p['pc:count']:,}")
print("type          :", p.get("pc:type"))
print("CRS (EPSG)    :", p.get("proj:epsg"))
print("dimensions    :", ", ".join(s["name"] for s in p.get("pc:schemas", []))[:90])

autzen-classified in autzen

assets:
  data         application/vnd.laszip+copc
               https://data.pointcloud.org/autzen/autzen-classified.copc.laz
  thumbnail    image/png
               https://data.pointcloud.org/autzen/stac/hero.png

points        : 61,201
type          : lidar
CRS (EPSG)    : None
dimensions    : X, Y, Z, Intensity, ReturnNumber, NumberOfReturns, ScanDirectionFlag, EdgeOfFlightLine, Cl


## 8. Read the point cloud without downloading it

COPC's whole point is range reads: a client fetches only the octree nodes it needs.
`item.assets["data"].href` is a plain HTTPS URL, so anything that speaks COPC — PDAL,
QGIS, the browser viewer on pointcloud.org — can open it in place.

The cell below reads just the **header** with PDAL, transferring a few kilobytes
rather than the whole file. It needs `pdal` installed (`conda install -c conda-forge
python-pdal`) and is skipped if that's missing.

In [12]:
href = item.assets["data"].href
print("COPC URL:", href)

try:
    import pdal
    pipeline = pdal.Pipeline(json.dumps({
        "pipeline": [{"type": "readers.copc", "filename": href, "count": 0}]
    }))
    pipeline.execute()
    meta = pipeline.metadata["metadata"]["readers.copc"]
    print()
    print("points in file:", f"{meta['count']:,}")
    print("bounds        :", {k: round(v, 2) for k, v in meta.items() if k in ("minx","miny","maxx","maxy")})
except ImportError:
    print()
    print("PDAL not installed — install python-pdal to run this, or try the URL directly:")
    print(f"  pdal info --summary {href}")

COPC URL: https://data.pointcloud.org/autzen/autzen-classified.copc.laz

PDAL not installed — install python-pdal to run this, or try the URL directly:
  pdal info --summary https://data.pointcloud.org/autzen/autzen-classified.copc.laz


## 9. Bulk queries: skip the API entirely

The API is convenient for a handful of results. For anything analytical — *every*
dataset's point count, grouped and aggregated — the archive also publishes its whole
catalog as a single
[STAC-GeoParquet](https://github.com/stac-utils/stac-geoparquet) file, which DuckDB
can query over HTTP without downloading it either.

This is the same data `/search` reads, so the two never disagree.

In [13]:
try:
    import duckdb
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    # DuckDB renders TIMESTAMPTZ in the session's timezone; pin it to UTC so these
    # values read the same as the API's.
    con.execute("SET TimeZone='UTC'")
    df = con.execute("""
        SELECT collection,
               "pc:count" AS points,
               CAST(datetime AS VARCHAR) AS acquired
        FROM read_parquet('https://data.pointcloud.org/stac/items.parquet')
        ORDER BY points DESC
        LIMIT 8
    """).df()
    print(df.to_string(index=False))
except ImportError:
    print("duckdb not installed — pip install duckdb to run this")

               collection       points               acquired
      MN_RainyLake_1_2020 355339247767 2021-04-16 00:00:00+00
  MN_UpperMissRiver_6_B22 325278739823 2022-05-23 00:00:00+00
      MN_RainyLake_2_2020 293472666706 2021-04-16 00:00:00+00
       MN_RiverWest_1_B23 288918266196 2023-05-04 00:00:00+00
       MN_RiverWest_2_B23 264904992691 2023-05-15 00:00:00+00
MN_CentralMissRiver_4_B22 215065503920 2022-05-04 00:00:00+00
  MN_UpperMissRiver_5_B22 200558106069 2022-06-09 00:00:00+00
      MN_GoodhueCo_1_2020 183448445150 2020-04-08 00:00:00+00


In [14]:
try:
    import duckdb
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    total = con.execute("""
        SELECT count(*) AS datasets, sum("pc:count") AS points
        FROM read_parquet('https://data.pointcloud.org/stac/items.parquet')
    """).fetchone()
    print(f"{total[0]} datasets, {total[1]:,} points in the archive")
except ImportError:
    print("duckdb not installed")

53 datasets, 4,944,371,387,959 points in the archive


## 10. Validating what you got

Anything the archive publishes should validate against the STAC schemas it declares.
`pystac` checks an Item against the core spec plus every extension in its
`stac_extensions` list, which is a quick way to confirm you're reading what you think
you are — and how the archive's own CI checks itself.

In [15]:
checked = 0
for item in client.search(max_items=10).items():
    item.validate()          # raises if the Item doesn't match its declared schemas
    checked += 1
print(f"{checked} items validated against their declared STAC schemas")

10 items validated against their declared STAC schemas


## Where to go next

- **Browse it**: [pointcloud.org](https://pointcloud.org) renders every dataset with an
  in-browser 3D viewer.
- **The raw catalog**: [`stac/catalog.json`](https://data.pointcloud.org/stac/catalog.json)
  is a static STAC Catalog, if you'd rather crawl than search.
- **QGIS**: add `https://pointcloud.org/stac` as a STAC connection.
- **Contribute a dataset**: one YAML manifest, one pull request —
  [contribution guide](https://github.com/hobuinc/pointcloud.org/tree/main/manifests).

The archive is a technology demonstration by [Hobu, Inc.](https://hobu.co), who
maintain PDAL, Entwine and the COPC specification.